# 06 · Model evaluation
**Objective:** read the held-out results, compare methods, and state what the evidence does and does not support.
Results come from `python -m app.agents.evaluation` runs stored in `research/evaluations/`; this notebook does not re-run them.

In [ ]:
# Setup: works locally and on Google Colab. Never put secrets in notebook cells.
import json, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = ""  # Set to your clone URL. On Colab, add HF_TOKEN under Secrets to use hosted models.
    if not Path("salesiq").exists():
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL in this cell to clone the SalesIQ repository.")
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, "salesiq"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "pyyaml", "pydantic", "mcp"], check=True)
    ROOT = Path("salesiq").resolve()
    from google.colab import userSecrets
    try:
        os.environ["HF_TOKEN"] = userSecrets.get("HF_TOKEN")
    except userSecrets.SecretNotFoundError:
        print("No HF_TOKEN secret; only a local model server can be used.")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research").is_dir())
sys.path.insert(0, str(ROOT / "apps" / "api"))
os.environ["SALESIQ_RESEARCH_DIR"] = str(ROOT / "research")

print('Reading stored results; no model is called here.')

In [ ]:
from IPython.display import Markdown, display
from app.agents.evaluation.report import load_summaries, merge, render

runs = load_summaries(split="test")
assert runs, "No held-out runs found. Run: python -m app.agents.evaluation --models baseline,qwen2.5-7b --split test"
merged = merge(runs)
display(Markdown(render(merged, runs)))

## Is any difference real?
Compare intervals before believing a gap. Non-overlap here is a rough screen, not a formal test.

In [ ]:
acc = {m: e["quality"] for m, e in merged["classification"].items()}
best = max(acc, key=lambda m: acc[m]["accuracy"])
base = acc["baseline"]
for m, q in sorted(acc.items(), key=lambda kv: -kv[1]["accuracy"]):
    lo, hi = q["accuracy_ci95"]
    clear = lo > base["accuracy_ci95"][1]
    note = "baseline" if m == "baseline" else ("clearly above baseline" if clear else "not clearly above baseline")
    print(f"{m:14} {q['accuracy']:.0%}  CI {lo:.0%}-{hi:.0%}  {note}")

**Limitations:** fictional data, rule-derived labels, one prompt version, small samples, no human ratings. Fine-tuning is not
justified by this evidence; the next improvements are retrieval, stronger validation, and real-document evaluation.